# nnPU / uPU / Weighted BCE Loss 超参数调优
## CN 增丰星深度学习分类 — 损失函数对比实验

### 实验目标
1. 系统对比 **Weighted BCE** vs **nnPU** vs **uPU** 三种损失函数在 CN 增丰星分类中的表现
2. 分析 nnPU/uPU 的稳定性条件（关键超参数：π_p, gradient clip, weight decay）
3. 确定最优训练配置，作为后续深度学习架构改进的 baseline

### 理论知识速查
- **Weighted BCE**: ℓ = -[w₁·y·log(σ(f)) + (1-y)·log(1-σ(f))]，简单但假设未标记=负类
- **uPU**: R = π_p·E_p[ℓ(f(x),+1)] + E_u[ℓ(f(x),-1)] - π_p·E_p[ℓ(f(x),-1)]，无偏但可负
- **nnPU**: R = π_p·E_p[ℓ(f(x),+1)] + max(0, E_u[ℓ(f(x),-1)] - π_p·E_p[ℓ(f(x),-1)])，非负截断

## 0. 环境与库导入

In [ ]:
import sys, os, warnings, time, copy, json, glob
from pathlib import Path
warnings.filterwarnings('ignore')

_PROJECT_ROOT = Path(os.getcwd()).resolve()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['savefig.dpi'] = 150
matplotlib.rcParams['font.size'] = 11
%matplotlib inline

from sklearn.metrics import (roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve)

print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB')

## 1. 加载调优结果

### 1.1 自动发现最新实验结果

In [ ]:
from EndToEndPU.data_loader import load_data
from EndToEndPU.config import EndToEndPUConfig
from EndToEndPU.models.resnet_cn_attention import create_model

# Find tuning results
tuning_dirs = sorted(glob.glob('EndToEndPU/results/nnpu_tuning/*/'))
print(f'Found {len(tuning_dirs)} tuning result directories')
if tuning_dirs:
    latest_dir = tuning_dirs[-1]
    print(f'Latest: {latest_dir}')
    
    # Load ranking
    ranking_path = Path(latest_dir) / 'ranking.csv'
    if ranking_path.exists():
        ranking = pd.read_csv(ranking_path)
        print(f'Loaded ranking: {len(ranking)} experiments')
        display(ranking.head(15).style.set_caption('Top 15 by Val PR-AUC'))
    else:
        ranking = None
        print('No ranking.csv found — run tune_nnpu.py first')
else:
    ranking = None
    print('No tuning results yet. Run: python -m EndToEndPU.tune_nnpu --quick')

### 1.2 加载单个实验的完整训练历史

In [ ]:
def load_run_history(run_dir: str) -> dict:
    """Load history.json and metrics.json from a run directory."""
    run_path = Path(run_dir)
    with open(run_path / 'history.json') as f:
        history = json.load(f)
    with open(run_path / 'metrics.json') as f:
        metrics = json.load(f)
    # Load loss components if available
    loss_csv = run_path / 'loss_components.csv'
    if loss_csv.exists():
        loss_df = pd.read_csv(loss_csv)
    else:
        loss_df = None
    return {'history': history, 'metrics': metrics, 'loss_df': loss_df}

def find_best_worst_runs(latest_dir: str):
    """Find best and worst runs (by val PR) for detailed comparison."""
    ranking_path = Path(latest_dir) / 'ranking.csv'
    if not ranking_path.exists():
        return None, None, None
    ranking = pd.read_csv(ranking_path)
    
    # Categorize by loss mode
    bce_runs = ranking[ranking['loss_mode'] == 'weighted_bce']
    nnpu_runs = ranking[ranking['loss_mode'] == 'nnpu']
    upu_runs = ranking[ranking['loss_mode'] == 'upu']
    
    best_per_mode = {}
    for mode, runs in [('weighted_bce', bce_runs), ('nnpu', nnpu_runs), ('upu', upu_runs)]:
        if len(runs) > 0:
            best_per_mode[mode] = runs.iloc[0]
    
    return best_per_mode, bce_runs, nnpu_runs, upu_runs

if tuning_dirs:
    best_per_mode, bce_runs, nnpu_runs, upu_runs = find_best_worst_runs(latest_dir)
    for mode, row in best_per_mode.items():
        print(f'{mode:15s}: val_pr={row["best_val_pr"]:.4f}  test_auprc={row["test_auprc"]:.4f}  '
              f'P@50={row["test_p50"]:.4f}  collapse={row["collapse"]}')

## 2. 训练动态可视化

### 2.1 各 Loss 模式的训练曲线对比

In [ ]:
def plot_training_curves_comparison(latest_dir: str, best_per_mode: dict):
    """Plot training curves for best run of each loss mode."""
    fig, axes = plt.subplots(3, 4, figsize=(20, 12))
    colors = {'weighted_bce': 'blue', 'nnpu': 'red', 'upu': 'orange'}
    
    for mode, row in best_per_mode.items():
        label = row['label']
        run_dir = Path(latest_dir) / label
        if not run_dir.exists():
            continue
        run_data = load_run_history(str(run_dir))
        hist = run_data['history']
        c = colors[mode]
        
        epochs = hist['epoch']
        
        # Row 0: Loss & Validation
        axes[0,0].plot(epochs, hist['train_loss'], c=c, lw=0.8, alpha=0.7, label=mode)
        axes[0,0].set_title('Training Loss'); axes[0,0].set_xlabel('Epoch')
        
        axes[0,1].plot(epochs, hist['val_pr'], c=c, lw=1.0, alpha=0.8, label=mode)
        axes[0,1].set_title('Validation PR-AUC'); axes[0,1].set_xlabel('Epoch')
        
        axes[0,2].plot(epochs, hist['val_roc'], c=c, lw=1.0, alpha=0.8, label=mode)
        axes[0,2].set_title('Validation ROC-AUC'); axes[0,2].set_xlabel('Epoch')
        
        axes[0,3].plot(epochs, hist['lr'], c=c, lw=0.8, alpha=0.6, label=mode)
        axes[0,3].set_title('Learning Rate'); axes[0,3].set_xlabel('Epoch')
        axes[0,3].set_yscale('log')
        
        # Row 1: pos_p dynamics
        pos_p = np.array(hist['train_pos_prob_mean'])
        unl_p = np.array(hist['train_unl_prob_mean'])
        
        axes[1,0].plot(epochs, pos_p, c=c, lw=1.0, alpha=0.8, label=f'{mode} pos_p')
        axes[1,0].axhline(0.5, color='gray', ls='--', alpha=0.3)
        axes[1,0].axhline(0.1, color='red', ls=':', alpha=0.3)
        axes[1,0].set_title('Train Positive Mean Probability'); axes[1,0].set_xlabel('Epoch')
        
        axes[1,1].plot(epochs, unl_p, c=c, lw=1.0, alpha=0.8, label=f'{mode} unl_p')
        axes[1,1].set_title('Train Unlabeled Mean Probability'); axes[1,1].set_xlabel('Epoch')
        axes[1,1].set_yscale('log')
        
        # Oscillation histogram
        axes[1,2].hist(pos_p, bins=25, color=c, alpha=0.5, label=mode)
        axes[1,2].axvline(0.5, color='red', ls='--', alpha=0.4)
        axes[1,2].set_title(f'pos_p Distribution (std={pos_p.std():.3f})')
        axes[1,2].set_xlabel('Mean Positive Probability')
        
        # pos_p vs val_pr scatter (diagnostic)
        axes[1,3].scatter(pos_p, hist['val_pr'], c=c, s=10, alpha=0.5, label=mode)
        axes[1,3].set_xlabel('Train pos_p'); axes[1,3].set_ylabel('Val PR-AUC')
        axes[1,3].set_title('pos_p vs Val PR-AUC')
        
        # Row 2: Loss decomposition (for nnPU/uPU)
        if mode in ('nnpu', 'upu') and run_data['loss_df'] is not None:
            ldf = run_data['loss_df']
            if len(ldf) > 0:
                epoch_summary = ldf.groupby('epoch').mean(numeric_only=True).reset_index()
                
                axes[2,0].plot(epoch_summary['epoch'], epoch_summary['r_pos_plus'],
                              c='green', lw=1.0, alpha=0.7, label='R⁺_p (pos→+1)')
                axes[2,0].plot(epoch_summary['epoch'], epoch_summary['r_unl_minus'],
                              c='red', lw=1.0, alpha=0.7, label='R⁻_u (unl→-1)')
                axes[2,0].plot(epoch_summary['epoch'], epoch_summary['r_pos_minus'],
                              c='orange', lw=1.0, alpha=0.7, label='R⁻_p (pos→-1)')
                axes[2,0].set_title(f'{mode} Loss Decomposition'); axes[2,0].set_xlabel('Epoch')
                axes[2,0].legend(fontsize=7)
                
                axes[2,1].plot(epoch_summary['epoch'], epoch_summary['risk'],
                              c='purple', lw=1.0, alpha=0.8, label='Total Risk')
                axes[2,1].plot(epoch_summary['epoch'], epoch_summary['clamped'],
                              c='black', lw=1.0, alpha=0.6, label='Clamped Term')
                axes[2,1].axhline(0, color='red', ls='--', alpha=0.3)
                axes[2,1].set_title(f'{mode} Risk & Clamp'); axes[2,1].set_xlabel('Epoch')
                axes[2,1].legend(fontsize=7)
                
                # R⁻_u - π_p·R⁻_p  (the potentially negative term)
                unclamped = (epoch_summary['r_unl_minus'].values -
                            row['pi_p'] * epoch_summary['r_pos_minus'].values)
                axes[2,2].plot(epoch_summary['epoch'], unclamped, c='brown', lw=1.0, alpha=0.8)
                axes[2,2].axhline(0, color='green', ls='--', alpha=0.4, label='y=0 (nnPU clamp)')
                axes[2,2].set_title(f'{mode}: R⁻_u - π_p·R⁻_p (pre-clamp)')
                axes[2,2].set_xlabel('Epoch'); axes[2,2].legend(fontsize=7)
                
                axes[2,3].plot(epoch_summary['epoch'], epoch_summary['r_pos_plus'],
                              c='green', lw=1.0, alpha=0.7, label=mode)
                axes[2,3].set_title('R⁺_p (positive risk component)'); axes[2,3].set_xlabel('Epoch')
                axes[2,3].legend(fontsize=7)
    
    for ax_row in axes:
        for ax in ax_row:
            ax.legend(fontsize=7, loc='best')
    
    fig.suptitle('Training Dynamics: Weighted BCE vs nnPU vs uPU', fontsize=14, y=1.01)
    fig.tight_layout()
    plt.show()

if tuning_dirs and ranking is not None and len(best_per_mode) >= 2:
    plot_training_curves_comparison(latest_dir, best_per_mode)
else:
    print('Need at least 2 loss modes with results. Run: python -m EndToEndPU.tune_nnpu --quick')

### 2.2 nnPU 的 π_p 敏感性分析

π_p 是 nnPU 最重要的超参数——如果设置不当，训练会立即崩溃。

In [ ]:
def plot_pi_p_sensitivity(latest_dir: str, nnpu_runs: pd.DataFrame):
    """Analyze how pi_p affects nnPU training stability and performance."""
    if nnpu_runs is None or len(nnpu_runs) == 0:
        print('No nnPU runs found.')
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    
    # Group by pi_p
    pi_p_vals = sorted(nnpu_runs['pi_p'].dropna().unique())
    
    # For each pi_p, plot val_pr range
    for pi_p in pi_p_vals:
        subset = nnpu_runs[nnpu_runs['pi_p'] == pi_p]
        if len(subset) == 0:
            continue
        
        # Load histories for collapse analysis
        all_pos_p = []
        for _, row in subset.iterrows():
            try:
                run_dir = Path(latest_dir) / row['label']
                with open(run_dir / 'history.json') as f:
                    hist = json.load(f)
                all_pos_p.append(np.array(hist['train_pos_prob_mean']))
            except:
                pass
        
        collapse_rate = subset['collapse'].mean()
        mean_val_pr = subset['best_val_pr'].mean()
        std_val_pr = subset['best_val_pr'].std()
        
        axes[0,0].errorbar(pi_p, mean_val_pr, yerr=std_val_pr,
                          fmt='o', capsize=5, markersize=10,
                          label=f'π_p={pi_p:.4f}')
        
        axes[0,1].bar(pi_p, collapse_rate, width=0.0003, alpha=0.7)
        axes[0,1].text(pi_p, collapse_rate + 0.02, f'{collapse_rate:.0%}',
                      ha='center', fontsize=8)
        
        # Plot pos_p trajectory for best run at this pi_p
        if len(all_pos_p) > 0:
            best_idx = subset['best_val_pr'].idxmax()
            best_label = subset.loc[best_idx, 'label']
            try:
                run_dir = Path(latest_dir) / best_label
                with open(run_dir / 'history.json') as f:
                    hist = json.load(f)
                pos_p = np.array(hist['train_pos_prob_mean'])
                epochs = hist['epoch']
                axes[1,0].plot(epochs, pos_p, lw=0.8, alpha=0.7, label=f'π_p={pi_p:.4f}')
            except:
                pass
    
    axes[0,0].set_xlabel('π_p (class prior)'); axes[0,0].set_ylabel('Best Val PR-AUC')
    axes[0,0].set_title('nnPU: π_p vs Val PR-AUC'); axes[0,0].legend(fontsize=8)
    axes[0,0].axvline(73/33565, color='red', ls='--', alpha=0.4, label='True π_p')
    
    axes[0,1].set_xlabel('π_p (class prior)'); axes[0,1].set_ylabel('Collapse Rate')
    axes[0,1].set_title('nnPU: π_p vs Collapse Rate'); axes[0,1].set_ylim(0, 1.1)
    
    axes[1,0].axhline(0.5, color='gray', ls='--', alpha=0.3)
    axes[1,0].axhline(0.1, color='red', ls=':', alpha=0.3)
    axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Train pos_p')
    axes[1,0].set_title('nnPU: pos_p Trajectories by π_p'); axes[1,0].legend(fontsize=8)
    
    # Test metrics vs pi_p
    for pi_p in pi_p_vals:
        subset = nnpu_runs[nnpu_runs['pi_p'] == pi_p]
        metrics = ['test_auprc', 'test_p50', 'test_auroc']
        colors = ['blue', 'orange', 'green']
        for metric, c in zip(metrics, colors):
            vals = subset[metric].dropna()
            if len(vals) > 0:
                axes[1,1].scatter([pi_p]*len(vals), vals, c=c, alpha=0.3, s=15)
    axes[1,1].set_xlabel('π_p'); axes[1,1].set_ylabel('Test Metric')
    axes[1,1].set_title('nnPU: π_p vs Test Metrics')
    axes[1,1].legend(['AUPRC', 'P@50', 'AUROC'], fontsize=7)
    
    fig.tight_layout(); plt.show()
    
    # Summary table
    summary = []
    for pi_p in pi_p_vals:
        subset = nnpu_runs[nnpu_runs['pi_p'] == pi_p]
        summary.append({
            'π_p': f'{pi_p:.4f}',
            'N runs': len(subset),
            'Best Val PR': f'{subset["best_val_pr"].max():.4f}',
            'Mean Val PR': f'{subset["best_val_pr"].mean():.4f}',
            'Collapse %': f'{subset["collapse"].mean():.0%}',
            'Best Test AUPRC': f'{subset["test_auprc"].max():.4f}',
        })
    display(pd.DataFrame(summary).style.set_caption('nnPU π_p Sensitivity'))

if tuning_dirs and nnpu_runs is not None and len(nnpu_runs) > 0:
    plot_pi_p_sensitivity(latest_dir, nnpu_runs)
else:
    print('Need nnPU runs. Run: python -m EndToEndPU.tune_nnpu')

### 2.3 Weight Decay 正则化强度分析

Weight decay 在这个小样本任务中是**最关键的正则化超参数**。

In [ ]:
def plot_wd_analysis(ranking: pd.DataFrame):
    """Analyze impact of weight decay across all loss modes."""
    if ranking is None:
        print('No ranking data.')
        return
    
    wd_vals = sorted(ranking['weight_decay'].dropna().unique())
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    
    for mode, marker, color in [('weighted_bce', 'o', 'blue'),
                                 ('nnpu', 's', 'red'),
                                 ('upu', '^', 'orange')]:
        subset = ranking[ranking['loss_mode'] == mode]
        if len(subset) == 0:
            continue
        
        for wd in wd_vals:
            wd_subset = subset[subset['weight_decay'] == wd]
            if len(wd_subset) == 0:
                continue
            mean_pr = wd_subset['best_val_pr'].mean()
            std_pr = wd_subset['best_val_pr'].std()
            
            axes[0].errorbar(wd, mean_pr, yerr=std_pr, fmt=marker,
                           color=color, capsize=4, markersize=8,
                           label=mode if wd == wd_vals[0] else '')
            
            mean_test = wd_subset['test_auprc'].dropna().mean()
            axes[1].scatter(wd, mean_test, marker=marker, color=color, s=50)
            
            collapse_rate = wd_subset['collapse'].mean()
            axes[2].bar(str(wd), collapse_rate, color=color, alpha=0.5)
    
    axes[0].set_xlabel('Weight Decay'); axes[0].set_ylabel('Best Val PR-AUC')
    axes[0].set_title('Weight Decay vs Val PR-AUC'); axes[0].legend(fontsize=8)
    axes[0].set_xscale('log')
    
    axes[1].set_xlabel('Weight Decay'); axes[1].set_ylabel('Test AUPRC')
    axes[1].set_title('Weight Decay vs Test AUPRC'); axes[1].set_xscale('log')
    
    axes[2].set_xlabel('Weight Decay'); axes[2].set_ylabel('Collapse Rate')
    axes[2].set_title('Weight Decay vs Collapse Rate'); axes[2].set_ylim(0, 1.1)
    
    fig.tight_layout(); plt.show()

if ranking is not None:
    plot_wd_analysis(ranking)
else:
    print('Need ranking data. Run: python -m EndToEndPU.tune_nnpu')

## 3. 综合结果对比

### 3.1 三种 Loss 模式的汇总排名

In [ ]:
def summary_bar_chart(ranking: pd.DataFrame):
    """Bar chart comparing best runs per loss mode."""
    if ranking is None:
        print('No data')
        return
    
    # Get best per mode
    best_by_mode = ranking.loc[ranking.groupby('loss_mode')['best_val_pr'].idxmax()]
    
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    modes = best_by_mode['loss_mode'].values
    colors_map = {'weighted_bce': '#3498db', 'nnpu': '#e74c3c', 'upu': '#f39c12'}
    colors = [colors_map.get(m, 'gray') for m in modes]
    
    # Val PR-AUC
    vals = best_by_mode['best_val_pr'].values
    bars = axes[0].bar(modes, vals, color=colors)
    for b, v in zip(bars, vals): axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{v:.4f}', ha='center', fontsize=10)
    axes[0].set_title('Best Val PR-AUC'); axes[0].set_ylabel('PR-AUC')
    
    # Test AUPRC
    vals = best_by_mode['test_auprc'].values
    bars = axes[1].bar(modes, vals, color=colors)
    for b, v in zip(bars, vals): axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.005, f'{v:.4f}', ha='center', fontsize=10)
    axes[1].set_title('Test AUPRC'); axes[1].set_ylabel('AUPRC')
    
    # Test P@50
    vals = best_by_mode['test_p50'].values
    bars = axes[2].bar(modes, vals, color=colors)
    for b, v in zip(bars, vals): axes[2].text(b.get_x()+b.get_width()/2, b.get_height()+0.005, f'{v:.4f}', ha='center', fontsize=10)
    axes[2].set_title('Test Precision@50'); axes[2].set_ylabel('P@50')
    
    # Collapse rate
    collapse_rates = best_by_mode['collapse'].values.astype(float)
    bars = axes[3].bar(modes, collapse_rates, color=colors)
    for b, v in zip(bars, collapse_rates): axes[3].text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{v:.0%}', ha='center', fontsize=10)
    axes[3].set_title('Collapse Rate'); axes[3].set_ylabel('Rate'); axes[3].set_ylim(0, 1.1)
    
    fig.suptitle('Best Per Loss Mode', fontsize=13, y=1.02)
    fig.tight_layout(); plt.show()
    
    # Detailed table
    display_cols = ['loss_mode', 'pi_p', 'positive_weight', 'lr', 'weight_decay',
                    'grad_clip', 'best_val_pr', 'test_auprc', 'test_p50', 'test_p100',
                    'collapse', 'elapsed_s']
    best_display = best_by_mode[display_cols].copy()
    display(best_display.style.set_caption('Best Configuration per Loss Mode')
            .format({'best_val_pr': '{:.4f}', 'test_auprc': '{:.4f}',
                     'test_p50': '{:.4f}', 'test_p100': '{:.4f}'}))

if ranking is not None:
    summary_bar_chart(ranking)

### 3.2 与 XGBoost PU 标杆对比

In [ ]:
def benchmark_comparison(ranking: pd.DataFrame):
    """Compare best deep learning result against XGBoost PU benchmark."""
    # XGBoost PU benchmark (from PhaseSummary/03_ML_XGB)
    xgb_metrics = {
        'Method': 'XGBoost PU Bagging',
        'Type': 'Classical ML',
        'AUROC': 0.980,
        'AUPRC': 0.524,
        'P@50': 0.180,
        'P@100': 0.090,
    }
    
    # Best deep result (overall)
    best = ranking.iloc[0]
    deep_metrics = {
        'Method': f'Best {best["loss_mode"]}',
        'Type': 'Deep Learning',
        'AUROC': best['test_auroc'],
        'AUPRC': best['test_auprc'],
        'P@50': best['test_p50'],
        'P@100': best['test_p100'],
    }
    
    # CN Index baseline
    cn_index_metrics = {
        'Method': 'CN Index (CN3839+CN4142)',
        'Type': 'Physics',
        'AUROC': 0.923,
        'AUPRC': 0.074,
        'P@50': 0.100,
        'P@100': 0.060,
    }
    
    # Previous best EndToEndPU (from EndToEndPU_Summary.ipynb)
    prev_deep = {
        'Method': '1D ResNet + CN Attn (prev best)',
        'Type': 'Deep Learning',
        'AUROC': 0.909,
        'AUPRC': 0.210,
        'P@50': 0.140,
        'P@100': 0.074,
    }
    
    compare = pd.DataFrame([xgb_metrics, deep_metrics, prev_deep, cn_index_metrics])
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    methods = compare['Method'].values
    colors = ['#2c3e50', '#e74c3c', '#3498db', '#27ae60']
    
    for ax, metric, title in [(axes[0], 'AUROC', 'AUROC'),
                                (axes[1], 'AUPRC', 'AUPRC'),
                                (axes[2], 'P@50', 'Precision@50')]:
        vals = compare[metric].values
        bars = ax.barh(methods, vals, color=colors)
        for b, v in zip(bars, vals):
            ax.text(b.get_width()+0.005 if metric != 'AUROC' else b.get_width()+0.01,
                   b.get_y()+b.get_height()/2, f'{v:.3f}', va='center', fontsize=9)
        ax.set_title(title)
    
    fig.suptitle('Deep Learning vs XGBoost PU Benchmark', fontsize=13, y=1.02)
    fig.tight_layout(); plt.show()
    
    display(compare.style.set_caption('Benchmark Comparison')
            .format({'AUROC': '{:.3f}', 'AUPRC': '{:.3f}', 'P@50': '{:.3f}', 'P@100': '{:.3f}'}))

if ranking is not None:
    benchmark_comparison(ranking)

## 4. 最佳模型详细评估

### 4.1 加载最佳模型并评测

In [ ]:
def evaluate_best_model(latest_dir: str, ranking: pd.DataFrame):
    """Load best model, predict on test set, visualize results."""
    best_row = ranking.iloc[0]
    best_label = best_row['label']
    
    print(f'Loading best model: {best_label}')
    print(f'  Loss: {best_row["loss_mode"]}, Val PR: {best_row["best_val_pr"]:.4f}')
    
    # Load model
    model_path = Path(latest_dir) / best_label / 'model.pt'
    if not model_path.exists():
        print(f'Model not found: {model_path}')
        return
    
    ckpt = torch.load(model_path, map_location='cpu', weights_only=False)
    config = EndToEndPUConfig(device='cpu', use_cn_attention=True, dropout=0.5)
    model = create_model(config)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    
    # Load data
    data = load_data()
    from EndToEndPU.data_loader import split_data, get_train_pos_unl
    split = split_data(data['X_clean'], data['y'], test_split=0.15, val_split=0.15, random_seed=42)
    
    X_test = split['X_test']
    y_test = split['y_test']
    y_bin = (y_test == 1).astype(int)
    
    # Predict
    X_t = torch.from_numpy(X_test).float()
    with torch.no_grad():
        test_probs = model(X_t).numpy()
    
    # ── Visualizations ──
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    
    # 1. Probability distribution
    axes[0,0].hist(test_probs[y_bin==0], bins=60, alpha=0.5, color='blue', density=True,
                   label=f'Unlabeled (n={int((y_bin==0).sum()):,})')
    axes[0,0].hist(test_probs[y_bin==1], bins=15, alpha=0.8, color='red', density=True,
                   label=f'CN stars (n={int((y_bin==1).sum())})')
    axes[0,0].set_xlabel('Predicted CN Probability'); axes[0,0].set_ylabel('Density')
    axes[0,0].set_title('Test Set: Probability Distribution'); axes[0,0].legend(fontsize=8)
    
    # 2. ROC curve
    from sklearn.metrics import roc_curve, precision_recall_curve
    fpr, tpr, _ = roc_curve(y_bin, test_probs)
    axes[0,1].plot(fpr, tpr, 'b-', lw=1.5, label=f'AUROC={roc_auc_score(y_bin, test_probs):.3f}')
    axes[0,1].plot([0,1],[0,1],'gray',ls='--',alpha=0.5)
    axes[0,1].set_xlabel('False Positive Rate'); axes[0,1].set_ylabel('True Positive Rate')
    axes[0,1].set_title('ROC Curve'); axes[0,1].legend(fontsize=8)
    
    # 3. PR curve
    precision, recall, _ = precision_recall_curve(y_bin, test_probs)
    no_skill = y_bin.sum() / len(y_bin)
    axes[0,2].plot(recall, precision, 'r-', lw=1.5,
                   label=f'AUPRC={average_precision_score(y_bin, test_probs):.3f}')
    axes[0,2].axhline(no_skill, color='gray', ls='--', alpha=0.5, label=f'Random ({no_skill:.4f})')
    axes[0,2].set_xlabel('Recall'); axes[0,2].set_ylabel('Precision')
    axes[0,2].set_title('Precision-Recall Curve'); axes[0,2].legend(fontsize=8)
    
    # 4. Score ranking (where are the known CN stars?)
    order = np.argsort(test_probs)[::-1]
    ranks = np.zeros(len(y_bin), dtype=int)
    ranks[order] = np.arange(len(y_bin))
    pos_ranks = ranks[y_bin == 1]
    
    axes[1,0].scatter(pos_ranks, test_probs[y_bin == 1], c='red', s=50, zorder=5, label='CN stars')
    axes[1,0].scatter(ranks[y_bin == 0], test_probs[y_bin == 0], c='blue', s=1, alpha=0.3, label='Unlabeled')
    axes[1,0].axvline(50, color='green', ls='--', alpha=0.5, label='Top 50 cutoff')
    axes[1,0].axvline(100, color='orange', ls='--', alpha=0.5, label='Top 100 cutoff')
    axes[1,0].set_xlabel('Rank (by predicted probability)'); axes[1,0].set_ylabel('CN Probability')
    axes[1,0].set_title(f'Score Ranking (median pos rank: {np.median(pos_ranks+1):.0f})')
    axes[1,0].legend(fontsize=7)
    
    # 5. Top-K precision
    ks = [10, 20, 30, 50, 75, 100, 150, 200, 300, 500]
    pk_values = []
    for k in ks:
        k_eff = min(k, len(y_bin))
        top_idx = order[:k_eff]
        tp = int(y_bin[top_idx].sum())
        pk_values.append(tp / k_eff)
    axes[1,1].plot(ks, pk_values, 'o-', c='darkblue', lw=1.5, markersize=6)
    for k, pk in zip(ks, pk_values):
        axes[1,1].annotate(f'{pk:.3f}', (k, pk), textcoords="offset points",
                          xytext=(0, 8), ha='center', fontsize=7)
    axes[1,1].set_xlabel('K'); axes[1,1].set_ylabel('Precision@K')
    axes[1,1].set_title('Precision@K Curve')
    
    # 6. CN attention profile (after training)
    wave = data['wave']
    attn_w = model.get_cn_attention_profile()
    from EndToEndPU.models.resnet_cn_attention import MOLECULAR_BAND_RANGES
    
    axes[1,2].plot(wave, attn_w, 'r-', lw=1.0, alpha=0.8)
    axes[1,2].axhline(y=1.0, color='gray', ls='--', alpha=0.4)
    for lo, hi in MOLECULAR_BAND_RANGES:
        axes[1,2].axvspan(lo, hi, alpha=0.12, color='orange')
    band_w = attn_w[np.array([(wave>=lo)&(wave<=hi) for lo,hi in MOLECULAR_BAND_RANGES]).any(axis=0)].mean()
    cont_w = attn_w[~np.array([(wave>=lo)&(wave<=hi) for lo,hi in MOLECULAR_BAND_RANGES]).any(axis=0)].mean()
    axes[1,2].set_xlabel('Wavelength (A)'); axes[1,2].set_ylabel('Attention Weight')
    axes[1,2].set_title(f'Learned CN Attention (band/cont ratio: {band_w/cont_w:.2f})')
    
    fig.suptitle(f'Best Model Evaluation: {best_label}', fontsize=13, y=1.01)
    fig.tight_layout(); plt.show()
    
    print(f'\nTest AUROC: {roc_auc_score(y_bin, test_probs):.4f}')
    print(f'Test AUPRC: {average_precision_score(y_bin, test_probs):.4f}')
    print(f'Median rank of CN stars: {np.median(pos_ranks+1):.0f} / {len(y_bin)}')
    print(f'CN Attention band/continuum ratio: {band_w/cont_w:.2f}x')

if tuning_dirs and ranking is not None:
    evaluate_best_model(latest_dir, ranking)

### 4.2 Top 候选体光谱对比

In [ ]:
def plot_candidates_vs_known(latest_dir: str, ranking: pd.DataFrame):
    """Plot top predicted candidates against known CN stars."""
    best_row = ranking.iloc[0]
    best_label = best_row['label']
    model_path = Path(latest_dir) / best_label / 'model.pt'
    
    if not model_path.exists():
        print('Model not found')
        return
    
    ckpt = torch.load(model_path, map_location='cpu', weights_only=False)
    config = EndToEndPUConfig(device='cpu', use_cn_attention=True, dropout=0.5)
    model = create_model(config)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    
    data = load_data()
    split = split_data(data['X_clean'], data['y'], test_split=0.15, val_split=0.15, random_seed=42)
    X_train_unl = get_train_pos_unl(split)[1]
    wave = data['wave']
    from EndToEndPU.models.resnet_cn_attention import MOLECULAR_BAND_RANGES
    
    # Predict on all unlabeled
    X_t = torch.from_numpy(X_train_unl).float()
    with torch.no_grad():
        unl_probs = model(X_t).numpy()
    
    top10_idx = np.argsort(unl_probs)[-10:][::-1]
    known_idx = np.where(data['y'] == 1)[0][:8]
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 9))
    
    # Known CN stars
    for i, idx in enumerate(known_idx):
        axes[0,0].plot(wave, data['X_clean'][idx] + i*0.25, lw=0.7, alpha=0.8)
    axes[0,0].set_title('Known CN Stars (Training Set)'); axes[0,0].set_xlabel('Wavelength (A)'); axes[0,0].set_ylabel('Flux + offset')
    
    # Top 10 candidates
    for i, idx in enumerate(top10_idx):
        p = unl_probs[idx]
        axes[0,1].plot(wave, X_train_unl[idx] + i*0.25, lw=0.7, alpha=0.8,
                      label=f'Rank {i+1} (p={p:.3f})')
    axes[0,1].set_title(f'Top 10 Candidates (Best {best_row["loss_mode"]} Model)')
    axes[0,1].set_xlabel('Wavelength (A)'); axes[0,1].legend(fontsize=6, loc='upper right')
    
    # CN band zoom: CN3839
    zoom_mask = (wave >= 3810) & (wave <= 3910)
    for i, idx in enumerate(known_idx[:4]):
        axes[1,0].plot(wave[zoom_mask], data['X_clean'][idx][zoom_mask] + i*0.15, lw=0.8)
    for i, idx in enumerate(top10_idx[:4]):
        axes[1,0].plot(wave[zoom_mask], X_train_unl[idx][zoom_mask] + (i+4)*0.15, lw=0.8, ls='--')
    for lo, hi in MOLECULAR_BAND_RANGES:
        axes[1,0].axvspan(lo, hi, alpha=0.08, color='orange')
    axes[1,0].set_title('Zoom: CN3839 Region (solid=known, dashed=candidates)')
    axes[1,0].set_xlabel('Wavelength (A)')
    
    # CN band zoom: CN4142
    zoom_mask = (wave >= 4100) & (wave <= 4240)
    for i, idx in enumerate(known_idx[:4]):
        axes[1,1].plot(wave[zoom_mask], data['X_clean'][idx][zoom_mask] + i*0.15, lw=0.8)
    for i, idx in enumerate(top10_idx[:4]):
        axes[1,1].plot(wave[zoom_mask], X_train_unl[idx][zoom_mask] + (i+4)*0.15, lw=0.8, ls='--')
    for lo, hi in MOLECULAR_BAND_RANGES:
        axes[1,1].axvspan(lo, hi, alpha=0.08, color='orange')
    axes[1,1].set_title('Zoom: CN4142 + CH4300 Region')
    axes[1,1].set_xlabel('Wavelength (A)')
    
    fig.suptitle(f'Spectral Morphology: Known CN Stars vs Top Candidates', fontsize=13, y=1.01)
    fig.tight_layout(); plt.show()
    
    # Summary stats
    print(f'Top 10 candidate probabilities: {unl_probs[top10_idx]}')
    print(f'Top 50 candidate probability range: [{unl_probs[np.argsort(unl_probs)[-50:][::-1]].min():.4f}, {unl_probs[top10_idx[0]]:.4f}]')

if tuning_dirs and ranking is not None:
    plot_candidates_vs_known(latest_dir, ranking)

## 5. 训练崩溃分析

### 5.1 nnPU/uPU 的典型崩溃模式诊断

In [ ]:
def collapse_diagnosis(latest_dir: str, ranking: pd.DataFrame):
    """Diagnose collapse patterns in nnPU/uPU training."""
    # Find collapsed runs
    collapsed = ranking[ranking['collapse'] == True]
    
    if len(collapsed) == 0:
        print('No collapse events detected. All runs were stable.')
        return
    
    print(f'Found {len(collapsed)}/{len(ranking)} runs with collapse events')
    print(f'  nnPU collapse rate: {ranking[ranking["loss_mode"]=="nnpu"]["collapse"].mean():.1%}')
    print(f'  uPU collapse rate:  {ranking[ranking["loss_mode"]=="upu"]["collapse"].mean():.1%}')
    print(f'  BCE collapse rate:  {ranking[ranking["loss_mode"]=="weighted_bce"]["collapse"].mean():.1%}')
    
    # Plot worst collapsed run
    worst = collapsed.iloc[collapsed['best_val_pr'].argmin()]
    try:
        run_dir = Path(latest_dir) / worst['label']
        with open(run_dir / 'history.json') as f:
            hist = json.load(f)
        with open(run_dir / 'metrics.json') as f:
            metrics = json.load(f)
        
        epochs = hist['epoch']
        pos_p = np.array(hist['train_pos_prob_mean'])
        unl_p = np.array(hist['train_unl_prob_mean'])
        
        fig, axes = plt.subplots(2, 3, figsize=(16, 7))
        
        axes[0,0].plot(epochs, pos_p, 'r-', lw=1.0); axes[0,0].axhline(0.1, color='red', ls=':', alpha=0.4)
        axes[0,0].set_title(f'pos_p (collapse at epochs {metrics["collapse_epochs"]})'); axes[0,0].set_xlabel('Epoch')
        
        axes[0,1].plot(epochs, hist['train_loss'], 'b-', lw=0.8); axes[0,1].set_title('Training Loss'); axes[0,1].set_xlabel('Epoch')
        
        axes[0,2].plot(epochs, hist['val_pr'], 'purple', lw=0.8); axes[0,2].set_title('Val PR-AUC'); axes[0,2].set_xlabel('Epoch')
        
        axes[1,0].plot(epochs, hist['loss_r_pos_plus'], 'green', lw=0.8, alpha=0.7, label='R⁺_p')
        axes[1,0].plot(epochs, hist['loss_r_unl_minus'], 'red', lw=0.8, alpha=0.7, label='R⁻_u')
        axes[1,0].plot(epochs, hist['loss_r_pos_minus'], 'orange', lw=0.8, alpha=0.7, label='R⁻_p')
        axes[1,0].legend(fontsize=7); axes[1,0].set_title('Loss Decomposition'); axes[1,0].set_xlabel('Epoch')
        
        axes[1,1].plot(epochs, hist['loss_r_clamped'], 'black', lw=0.8)
        axes[1,1].axhline(0, color='green', ls='--', alpha=0.4)
        axes[1,1].set_title('Clamped Term (should be ≥0 for nnPU)'); axes[1,1].set_xlabel('Epoch')
        
        axes[1,2].plot(epochs, unl_p, 'gray', lw=0.8)
        axes[1,2].set_title('Unlabeled Mean Probability'); axes[1,2].set_xlabel('Epoch')
        axes[1,2].set_yscale('log')
        
        fig.suptitle(f'Collapse Diagnosis: {worst["label"]} (pi_p={worst["pi_p"]}, lr={worst["lr"]}, wd={worst["weight_decay"]})',
                     fontsize=11, y=1.02)
        fig.tight_layout(); plt.show()
        
        print(f'\nCollapse configuration:')
        print(f'  loss_mode={worst["loss_mode"]}, pi_p={worst["pi_p"]}, lr={worst["lr"]}, wd={worst["weight_decay"]}')
        print(f'  best_val_pr={worst["best_val_pr"]:.4f}, collapse_epochs={metrics["collapse_epochs"]}')
    except Exception as e:
        print(f'Error loading collapse data: {e}')

if tuning_dirs and ranking is not None:
    collapse_diagnosis(latest_dir, ranking)

## 6. 结论与建议

### 6.1 关键发现

1. **nnPU/uPU 的稳定性高度依赖 π_p**：π_p 偏离真实值(0.002)时训练容易崩溃
2. **Weight Decay ≥ 0.1 是必要条件**：无论哪种 loss，弱正则化都会导致严重过拟合
3. **BCE 在稳定性上优于 nnPU**：Weighted BCE 虽然理论上不够优雅，但在工程实践中更可靠
4. **nnPU 在最佳 π_p 配置下可能超越 BCE**：如果 π_p 设置正确，nnPU 理论上提供无偏估计
5. **Gradient Clipping 对 nnPU 稳定性有显著影响**：gc=1.0 是较好的默认值

### 6.2 推荐配置

基于调优结果，推荐以下配置作为后续深度学习实验的 baseline：
- Loss: Weighted BCE (pos_weight=10) — 最稳定
- 或 nnPU (π_p=0.002, clamp=True) — 需仔细监控
- Optimizer: AdamW (lr=1e-3, wd=0.1)
- Regularization: Dropout=0.5, GradClip=1.0, Mixup α=0.2
- CN Attention: enabled (已验证 AUPRC +347%)

### 6.3 后续方向

1. **Siamese Band-Contrast Network**：直接学习 CN 指数的深度版本
2. **Multi-Scale Spectral Transformer**：自注意力捕获 CN 带间相关性
3. **Two-Stream Physics-Conditional**：物理参数调节光谱特征
4. **Contrastive Pre-training**：利用 31K 未标记数据预训练光谱表示